In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ydata_profiling
import random
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import matthews_corrcoef as mcc, confusion_matrix, make_scorer
import os
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.inspection import permutation_importance, PartialDependenceDisplay


#### **Set the path**

In [ ]:
filepath = "path/to/data.csv"
data = pd.read_csv(filepath, sep=";") # <---read the csv in tne filepath
data.columns

#### **Filter and downsample the dataset**

In [ ]:

filter_data = data.query('gender in [1,2]')
filter_data.shape
downsampled_data = filter_data.sample(frac=0.25,random_state=42) # <-- select a fraction of the dataset
downsampled_data.shape


In [ ]:
downsampled_data.columns

In [ ]:
#### **Select and Split the features**

In [ ]:
features = ["Q1", "Q2", "Q3", "Q4", "Q5", "Q9", "Q12", "Q14", "Q15", "Q16","Q17","Q20","Q22","Q26","Q27","Q28","Q30","Q34","Q35","Q36","Q38","Q39","Q40","Q42","Q43","Q44"]


In [ ]:
target = ["gender"]

In [ ]:
X = downsampled_data[features].values
Y = downsampled_data[target].values

In [ ]:
X

In [ ]:
X = pd.DataFrame(downsampled_data[features].values)
Y = pd.DataFrame(downsampled_data[target].values)
corr = abs(X.corr())

In [ ]:
plt.imshow(corr)
plt.colorbar()
plt.xticks(np.arange(len(X.columns)))
plt.yticks(np.arange(len(X.columns)));

In [ ]:
X = downsampled_data[features].values
Y = downsampled_data[target].values

In [ ]:
X_train_not_normalized, X_test_not_normalized, Y_train, Y_test = train_test_split(X, Y, test_size = 0.25)

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train_not_normalized)
X_train = scaler.transform(X_train_not_normalized)
X_test = scaler.transform(X_test_not_normalized)

In [ ]:
Y_train.shape
Y_train = Y_train.ravel()
original_Y_test = Y_test

In [ ]:
Y_test

In [ ]:
original_Y_test

#### 5-fold partitioning

In [ ]:
n_folds = 5
kfold = StratifiedKFold(n_splits=n_folds)

#### Optimization of the model parameter

In [ ]:
MCC_all = []
C_list = [10.0**(x) for x in np.arange(-5, 5)]

for C in C_list:
    model = LogisticRegression(C=C)
    
    MCC_this_C = []
    for idx_fit, idx_valid in kfold.split(X_train, Y_train):
        X_fit = X_train[idx_fit, :]
        Y_fit = Y_train[idx_fit]
        
        X_valid = X_train[idx_valid, :]
        Y_valid = Y_train[idx_valid]
    
        model.fit(X_fit, Y_fit)
        Y_pred_valid = model.predict(X_valid)
        MCC = mcc(Y_valid, Y_pred_valid)
        MCC_this_C.append(MCC)
        
    MCC_all.append(MCC_this_C)
        
MCC_all = np.array(MCC_all)

In [ ]:
fig, ax = plt.subplots()
plt.plot(MCC_all, 'o-')
plt.ylabel('MCC')
plt.xlabel('C')
plt.xticks(np.arange(len(C_list)), C_list);
plt.legend(np.arange(n_folds))
plt.grid()

In [ ]:
C_best = 0.001 # <---manually set the best C
model_best = LogisticRegression(C=C_best)

#### Fit

In [ ]:
model_best.fit(X_train, Y_train)

#### Evaluate the performance

In [ ]:
Y_pred_train = model_best.predict(X_train)
Y_pred_test = model_best.predict(X_test)

print(mcc(Y_train, Y_pred_train))
print(confusion_matrix(Y_train, Y_pred_train))



print(mcc(Y_test, Y_pred_test))
print(confusion_matrix(Y_test, Y_pred_test))

#### Bootstrapping and confidence interval

In [ ]:
subset_ratio = 0.25
n_repeats = 100

MCC_train = []
MCC_test = []

n_train = X_train.shape[0]
n_test = X_test.shape[0]

#repeat 'n_repeats' times
for i in range(n_repeats):
    #select a subset
    idx_train = np.random.choice(n_train, int(subset_ratio*n_train))
    idx_test  = np.random.choice(n_test,  int(subset_ratio*n_test ))
    
    #generate prediction on the subset
    Y_pred_train_ = model_best.predict(X_train[idx_train])
    Y_pred_test_ = model_best.predict(X_test[idx_test])
    
    #compute (and store) performances on the subset
    MCC_train.append(mcc(Y_train[idx_train], Y_pred_train_))
    MCC_test.append(mcc(Y_test[idx_test], Y_pred_test_))
    
    
train_median, train_CI5, train_CI95 = np.quantile(MCC_train, [0.5, 0.05, 0.95])
test_median, test_CI5, test_CI95 = np.quantile(MCC_test, [0.5, 0.05, 0.95])




In [ ]:
print('MCC train: {:.3f} [{:.3f}-{:.3f}]'.format(train_median, train_CI5, train_CI95))
print('MCC test: {:.3f} [{:.3f}-{:.3f}]'.format(test_median, test_CI5, test_CI95))

In [ ]:
plt.bar(0, train_median)
plt.plot([0,0], [train_CI5, train_CI95], 'k')

plt.bar(1, test_median)
plt.plot([1,1], [test_CI5, test_CI95], 'k')

plt.xticks([0,1], ['Train', 'Test'])

#### Features importance

In [ ]:
r = permutation_importance(model_best, X_test, Y_test,
                           n_repeats=30,
                           scoring=make_scorer(mcc))

In [ ]:
fig, ax = plt.subplots()
ax.boxplot(
    r.importances.T, vert=True
);

In [ ]:
result = permutation_importance(model_best, X_test, Y_test, n_repeats=10,
                                random_state=42, n_jobs=2)
sorted_idx = result.importances_mean.argsort()[-10:]

X_test =pd.DataFrame(X_test)
fig, ax = plt.subplots()
ax.barh(range(len(sorted_idx)), result.importances[sorted_idx].mean(axis=1).T)
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels(X_test.columns[sorted_idx])
ax.set_title("Permutation Importances (test set)")
fig.tight_layout()
plt.show()

#### Diagnostic test

In [ ]:
from sklearn.svm import SVC #<-- testing another classification methods 

In [ ]:
C_best = 0.001
kernel_best = 'linear'
model_best = SVC(C=C_best, kernel=kernel_best)
model_best.fit(X_train, Y_train)

Y_pred_train = model_best.predict(X_train)
Y_pred_test = model_best.predict(X_test)

print(mcc(Y_train, Y_pred_train))
print(confusion_matrix(Y_train, Y_pred_train))

print(mcc(original_Y_test, Y_pred_test))
print(confusion_matrix(original_Y_test, Y_pred_test))

In [ ]:
random.shuffle(Y_test) # <-- Random labels method - Shuffle the target variable  


In [ ]:
Y_pred_train = model_best.predict(X_train)
Y_pred_test = model_best.predict(X_test)

print(mcc(Y_train, Y_pred_train))
print(confusion_matrix(Y_train, Y_pred_train))

print(mcc(Y_test, Y_pred_test)) # <-- mcc calculated with shuffled values of Y_test
print(confusion_matrix(Y_test, Y_pred_test))